# 🏰 Citadel Backend Contracts - Testing Center

## Explore How to Bring New AI Backends/Models into the Citadel Governance Hub!

Use this Jupyter notebook with Python code snippets to onboard all your AI backends and all associated routing logic, including:
- Configuring LLM backend endpoints and authentication
- Generating the stack variables file (`environments/<env>/llm-backend-onboarding.tfvars`)
- Deploying backend pools and policy fragments with the `stacks/llm-backend-onboarding` Terraform stack
- Testing deployed models through multiple API formats
- Verifying streaming and SDK integration

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Governance Hub Deployment Guide](../guides/full-deployment-guide.md) or [Citadel Governance Hub Quick Deployment Guide](../guides/quick-deployment-guide.md) before proceeding.

## Azure Prerequisites

- An existing Citadel Governance Hub deployment with APIM configured
- LLM backend endpoints (AI Foundry, Azure OpenAI, or external) provisioned and accessible
- Managed Identity configured for APIM authentication to backend services
- `environments/<env>/common.tfvars` and `backend.hcl` in place (see [`../examples/quickstart`](../examples/quickstart/README.md) and `task bootstrap ENV=<env>`)
- Terraform >= 1.5 and Azure CLI installed and authenticated (`az login`); [Task](https://taskfile.dev) is used when installed

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Start by defining the variables that will drive the configuration of your Access Contracts and the generation of the APIM product policies. Adjust these variables according to your testing needs.

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# 🌍 ENVIRONMENT: environments/<env>/ (common.tfvars, backend.hcl, <stack>.tfvars)
# ============================================================================
env       = "dev"                       # environment folder under environments/
repo_root = os.path.abspath("..")       # repository root (Taskfile.yml, stacks/, environments/)
env_dir   = utils.env_dir(env, repo_root)

# Var file generated for stacks/llm-backend-onboarding (next to common.tfvars).
# The default is the environment's own stack file (backed up to .bak the first time);
# use e.g. "llm-backend-onboarding.notebook.tfvars" to keep that file untouched.
llm_backend_tfvars_name = "llm-backend-onboarding.tfvars"

# ============================================================================
# REQUIRED: Update these values for your environment ("REPLACE" = read from the stacks)
# ============================================================================
governance_hub_resource_group = "REPLACE"          # Resource group of the deployed Citadel Governance Hub
location                      = "REPLACE"          # Azure region (e.g. "swedencentral", "eastus")

# ============================================================================
# LLM BACKENDS SOURCE
#   "foundry"  = derive one backend per platform Foundry account (llm_backends_config ignored)
#   "extra"    = Foundry-derived backends + llm_backends_config (extra_llm_backends)
#   "override" = llm_backends_config only (llm_backend_config, full override)
# ============================================================================
backend_mode       = "override"
inference_api_type = "OpenAIV1"   # AzureOpenAI | AzureAI | OpenAI | OpenAIV1
features           = {}           # e.g. {"unified_ai_api": True, "api_center_onboarding": True}

## ============================================================================
## UPDATE THIS CONFIGURATION BASED ON YOUR DEPLOYED BACKENDS
## This is an example configuration for two Azure AI Foundry backends. Update the values to match your deployed backends and models.

llm_backends_config = [
  {
    "authScheme": "managedIdentity",
    "authType": "managed-identity",
    "backendId": "aif-citadel-primary",
    "backendType": "ai-foundry",
    "endpoint": "https://REPLACE.services.ai.azure.com/",
    "priority": 1,
    "supportedModels": [
      {
        "capacity": 50,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-05",
        "name": "gpt-5.4",
        "retirementDate": "2027-03-05",
        "sku": "GlobalStandard",
        "timeout": 180
      },
      {
        "capacity": 1,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-17",
        "name": "gpt-5.4-mini",
        "retirementDate": "2027-09-21",
        "sku": "GlobalStandard"
      }
      
    ],
    "weight": 100
  }
]


# ============================================================================
# OPTIONAL: Model Aliases
# ----------------------------------------------------------------------------
# Group multiple models under a single client-facing name. Clients send the
# alias as the `model` value (or `/deployments/{alias}/...` URL); the gateway
# resolves it to a real model. The same alias map is honored consistently
# across Azure OpenAI API, Universal LLM API, and Unified AI API.
#
#   strategy: 'priority'  → first model wins; subsequent models act as
#                           cross-model fallback in the Unified AI API retry.
#             'weighted'  → random selection by weights (A/B testing or
#                           controlled rollouts). Requires `weights`.
#
# Direct model selection is unaffected — when `model` does NOT match an alias,
# routing falls back to the standard model→backend pool lookup.
#
# The defaults below are aligned with the accelerator's main `main.bicepparam`
# model line-up (gpt-4.1, gpt-5.4-mini, gpt-5.2, DeepSeek-R1, Phi-4,
# Mistral-Large-3, text-embedding-3-large). Adjust the `models` lists to
# match the real models you deploy.
# ============================================================================
# model_aliases = [
#     # Most-advanced GPT — automatically falls back when the top model is throttled.
#     { "name": "adv-gpt",        "models": ["gpt-5.4", "gpt-4.1"], "strategy": "priority" },
#     # Cost/latency-optimized GPT — prefers the smaller model with full-size fallback.
#     { "name": "fast-gpt",       "models": ["gpt-4.1-mini"],            "strategy": "priority" },
#     # Reasoning workloads — DeepSeek as primary, GPT-5.2 as fallback.
#     { "name": "reasoning",      "models": ["gpt-5.4"],             "strategy": "priority" },
#     # Stable embeddings alias — easy to swap underlying model later without
#     # touching client code.
#     { "name": "embeddings",     "models": ["text-embedding-3-small"],             "strategy": "priority" },
#     # Weighted A/B blend: 80% to the cheaper model, 20% to the larger one.
#     # Useful to validate gpt-4.1 quality on a slice of production traffic.
#     { "name": "ab-test-gpt",    "models": ["gpt-4.1-mini", "gpt-4.1"],            "strategy": "weighted", "weights": [80, 20] },
# ]
model_aliases = [
  { "name": "adv-gpt",        "models": ["gpt-5.4"], "strategy": "priority" },
  { "name": "mini-gpt",        "models": ["gpt-5.4-mini"], "strategy": "priority" },
  ]  # REPLACE: add your model aliases here, or leave empty to disable

# Backends whose authConfig uses `keyVaultSecretUri` need the APIM managed identity
# to hold `Key Vault Secrets User` on that vault (the platform Key Vault already does).

# ============================================================================
# OPTIONAL: AWS credentials (only required when an aws-bedrock backend is used)
# Versionless Key Vault secret URIs; APIM named values aws-access-key / aws-secret-key
# reference them (no plaintext keys). aws-region is a plain named value.
# ============================================================================
aws_access_key_secret_uri = ""  # e.g. https://<kv>.vault.azure.net/secrets/aws-access-key
aws_secret_key_secret_uri = ""  # e.g. https://<kv>.vault.azure.net/secrets/aws-secret-key
aws_region = ""

# Resolve "REPLACE" values from the deployed stacks (platform outputs / common.tfvars)
if governance_hub_resource_group in ("", "REPLACE"):
    governance_hub_resource_group = utils.terraform_output_get("resource_group_name", "REPLACE", stack="platform", env=env, repo_root=repo_root)
if location in ("", "REPLACE"):
    location = utils.read_common_tfvars(env, repo_root).get("location", location)
for _f in ("common.tfvars", "backend.hcl"):
    if not os.path.isfile(os.path.join(env_dir, _f)):
        utils.print_warning(f"Missing {os.path.join(env_dir, _f)} (see examples/quickstart and `task bootstrap ENV={env}`)")

utils.print_info(f"Environment: {env} ({env_dir})")
utils.print_info(f"Resource group: {governance_hub_resource_group} | Location: {location}")
utils.print_info(f"Initialization completed.")

<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [5]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:02:38.414911 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [6]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:02:40.962534 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476
⚙️ Running: az resource list -g rg-citadel-italynorth-01 --resource-type Microsoft.ApiManagement/service 
✅ Listing APIM Resources ⌚ 14:02:44.629058 :3s]
👉🏽 APIM Service Id: /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: 854bc984-355a-4e87-ae7d-1d0eedab998d
✅ APIM Client Tool initialized successfully! ⌚ 14:02:47.644140 
👉🏽 APIM Resource Name: apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net


<a id='3'></a>
### 3️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [7]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

👉🏽 Extracting current APIM backends configuration...
👉🏽 Retrieving APIM backends using Azure REST API...
👉🏽 🔗 Backend: content-safety-backend -> https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
👉🏽 🔗 Backend: foundry-italynorth-0 -> https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
👉🏽 🔗 Backend: ms-learn-mcp-backend -> https://learn.microsoft.com/api/mcp
✅ Found 3 individual backends and 0 backend pools ⌚ 14:25:25.585936 


In [8]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 2 unique supported models
✅ Supported models in APIM policy fragment 'set-backend-pools': ⌚ 14:25:32.342173 
  • gpt-5.4
  • gpt-5.4-mini


In [9]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • content-safety-backend
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
  • foundry-italynorth-0
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
    Models: gpt-5.4, gpt-5.4-mini
  • ms-learn-mcp-backend
    URL: https://learn.microsoft.com/api/mcp

🤖 Total Supported Models: 2
   gpt-5.4, gpt-5.4-mini


<a id='4'></a>
### 4️⃣ Discover Managed Identity for APIM Authentication

Display the user-assigned managed identity APIM uses to authenticate to managed-identity backends. This is informational: the `llm-backend-onboarding` stack finds the identity (like APIM itself) by its deterministic name from `common.tfvars`, so it is not a Terraform input:

In [ ]:
# Discover managed identity from APIM using the SDK (informational — the stack resolves it by name)
utils.print_info("Discovering managed identity configuration...")

managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name')
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover the APIM managed identity (managed-identity backends depend on it).")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

<a id='5'></a>
### 5️⃣ Generate LLM Backend Terraform Variables File

Generate `environments/<env>/llm-backend-onboarding.tfvars` (or the file named by `llm_backend_tfvars_name`) for the [`../stacks/llm-backend-onboarding`](../stacks/llm-backend-onboarding) stack. APIM, Key Vault and the identities are found by name from `environments/<env>/common.tfvars`, so the file only holds the backends (`foundry_backends` / `llm_backend_config` / `extra_llm_backends`), `model_aliases`, `features` and the optional `aws` settings:

In [11]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    auth = backend.get('authType') or backend.get('authScheme') or 'unspecified'
    print(f"     Auth: {auth}")
    auth_config = backend.get('authConfig')
    if auth_config:
        nv = auth_config.get('namedValueKey', 'N/A')
        if auth_config.get('keyVaultSecretUri'):
            print(f"     Auth Source: Key Vault → {auth_config['keyVaultSecretUri']} (named value: {nv})")
        elif auth_config.get('secretValue'):
            print(f"     Auth Source: inline secret (named value: {nv}) ⚠️ testing only")
        else:
            print(f"     Auth Source: named value '{nv}'")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            extras = []
            if 'retirementDate' in model:
                extras.append(f"Retirement: {model['retirementDate']}")
            if 'apiVersion' in model:
                extras.append(f"API: {model['apiVersion']}")
            if 'timeout' in model:
                extras.append(f"Timeout: {model['timeout']}s")
            if 'inferenceApiVersion' in model:
                extras.append(f"Inference API: {model['inferenceApiVersion']}")
            extra_str = f", {', '.join(extras)}" if extras else ""
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver}{extra_str})")
        else:
            print(f"       - {model_name}")

# Display model aliases if defined
if model_aliases:
    utils.print_info(f"\nModel Aliases ({len(model_aliases)}):")
    for alias in model_aliases:
        strategy = alias.get('strategy', 'priority')
        models_str = ', '.join(alias.get('models', []))
        weights = alias.get('weights')
        weights_str = f" weights={weights}" if weights else ""
        print(f"  • {alias['name']} → [{models_str}]  (strategy: {strategy}{weights_str})")


👉🏽 LLM Backends to be deployed:

  🔗 aif-citadel-primary
     Type: ai-foundry
     Endpoint: https://aif-citadel-italynorth-01-0-uv5wjw.services.ai.azure.com/
     Auth: managed-identity
     Priority: 1, Weight: 100
     Models (2):
       - gpt-5.4 (SKU: GlobalStandard, Capacity: 50, Format: OpenAI, Version: 2026-03-05, Retirement: 2027-03-05, Timeout: 180s)
       - gpt-5.4-mini (SKU: GlobalStandard, Capacity: 1, Format: OpenAI, Version: 2026-03-17, Retirement: 2027-09-21)
👉🏽 
Model Aliases (2):
  • adv-gpt → [gpt-5.4]  (strategy: priority)
  • mini-gpt → [gpt-5.4-mini]  (strategy: priority)


In [ ]:
# Generate the var file for the llm-backend-onboarding stack:
#   environments/<env>/<llm_backend_tfvars_name>  (common.tfvars supplies workload, environment,
#   subscription, location and naming, from which the stack finds APIM, Key Vault and identities)

stack_path = utils.stack_dir("llm-backend-onboarding", repo_root)
if not os.path.isdir(stack_path):
    utils.print_error(f"Terraform stack directory not found: {stack_path}")
tfvars_file = os.path.join(env_dir, llm_backend_tfvars_name)


def hcl_str(v):
    """Quote and escape a string value for HCL."""
    return '"' + str(v).replace("\\", "\\\\").replace('"', '\\"') + '"'

# Format a single model object for Terraform (HCL).
# Model field names stay camelCase to match the Terraform variable schema
# (name, sku, capacity, modelFormat, modelVersion, ...).
def format_model_for_tf(model):
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name = {hcl_str(model)} }}"

    parts = [f"name = {hcl_str(model['name'])}"]
    if 'sku' in model:                 parts.append(f"sku = {hcl_str(model['sku'])}")
    if 'capacity' in model:            parts.append(f"capacity = {model['capacity']}")
    if 'modelFormat' in model:         parts.append(f"modelFormat = {hcl_str(model['modelFormat'])}")
    if 'modelVersion' in model:        parts.append(f"modelVersion = {hcl_str(model['modelVersion'])}")
    if 'retirementDate' in model:      parts.append(f"retirementDate = {hcl_str(model['retirementDate'])}")
    if 'apiVersion' in model:          parts.append(f"apiVersion = {hcl_str(model['apiVersion'])}")
    if 'timeout' in model:             parts.append(f"timeout = {model['timeout']}")
    if 'inferenceApiVersion' in model: parts.append(f"inferenceApiVersion = {hcl_str(model['inferenceApiVersion'])}")
    return "{ " + ", ".join(parts) + " }"

# Map the notebook's camelCase authConfig keys to the Terraform snake_case schema.
def format_auth_config_for_tf(auth_config):
    parts = []
    if auth_config.get('namedValueKey'):
        parts.append(f"named_value_key = {hcl_str(auth_config['namedValueKey'])}")
    if auth_config.get('keyVaultSecretUri'):
        parts.append(f"key_vault_secret_uri = {hcl_str(auth_config['keyVaultSecretUri'])}")
    if auth_config.get('secretValue'):
        parts.append(f"secret_value = {hcl_str(auth_config['secretValue'])}")
    return "{ " + ", ".join(parts) + " }"

# Format a backend object for Terraform (snake_case keys, per-model metadata).
def format_backend_for_tf(backend):
    models_formatted = [format_model_for_tf(m) for m in backend['supportedModels']]
    # NOTE: HCL list items MUST be comma-separated. Join with a comma so the
    #       generated supported_models = [ ... ] block is valid HCL.
    models_str = ",\n      ".join(models_formatted)

    lines = [
        "  {",
        f"    backend_id   = {hcl_str(backend['backendId'])}",
        f"    backend_type = {hcl_str(backend['backendType'])}",
        f"    endpoint     = {hcl_str(backend['endpoint'])}",
    ]
    # Auth: carry over legacy authScheme and/or new authType
    if 'authScheme' in backend:
        lines.append(f"    auth_scheme  = {hcl_str(backend['authScheme'])}")
    if 'authType' in backend:
        lines.append(f"    auth_type    = {hcl_str(backend['authType'])}")
    if backend.get('authConfig'):
        lines.append(f"    auth_config  = {format_auth_config_for_tf(backend['authConfig'])}")
    lines.append(f"    supported_models = [\n      {models_str}\n    ]")
    lines.append(f"    priority = {backend.get('priority', 1)}")
    lines.append(f"    weight   = {backend.get('weight', 100)}")
    lines.append("  }")
    return "\n".join(lines)

# Format a model alias object for Terraform.
def format_alias_for_tf(alias):
    parts = [f"name = {hcl_str(alias['name'])}"]
    models_arr = ", ".join([hcl_str(m) for m in alias.get('models', [])])
    parts.append(f"models = [ {models_arr} ]")
    if 'strategy' in alias:
        parts.append(f"strategy = {hcl_str(alias['strategy'])}")
    if 'weights' in alias:
        weights_arr = ", ".join([str(w) for w in alias['weights']])
        parts.append(f"weights = [ {weights_arr} ]")
    return "  { " + ", ".join(parts) + " }"


# NOTE: HCL list items MUST be comma-separated. Join backends with a comma too.
backends_tf_str = ",\n".join([format_backend_for_tf(b) for b in llm_backends_config])

# Backends: Foundry-derived, Foundry-derived + extras, or a full override
if backend_mode == "foundry":
    backends_block = "foundry_backends = { enabled = true }\n"
elif backend_mode == "extra":
    backends_block = f"foundry_backends = {{ enabled = true }}\n\nextra_llm_backends = [\n{backends_tf_str}\n]\n"
else:  # override
    backends_block = f"foundry_backends = {{ enabled = false }}\n\nllm_backend_config = [\n{backends_tf_str}\n]\n"

# Model aliases block
if model_aliases:
    aliases_str = ",\n".join([format_alias_for_tf(a) for a in model_aliases])
    aliases_block = f"model_aliases = [\n{aliases_str}\n]\n"
else:
    aliases_block = "model_aliases = []\n"

# Optional features + AWS credential blocks
optional_block = ""
if features:
    feature_lines = "\n".join(f"  {k} = {str(bool(v)).lower()}" for k, v in features.items())
    optional_block += f"\n# Optional LLM APIs\nfeatures = {{\n{feature_lines}\n}}\n"
if aws_access_key_secret_uri or aws_secret_key_secret_uri or aws_region:
    optional_block += f"\n# AWS credentials for Amazon Bedrock (SigV4) backends (versionless Key Vault secret URIs)\n"
    optional_block += "aws = {\n"
    optional_block += f"  region                = {hcl_str(aws_region)}\n"
    optional_block += f"  access_key_secret_uri = {hcl_str(aws_access_key_secret_uri)}\n"
    optional_block += f"  secret_key_secret_uri = {hcl_str(aws_secret_key_secret_uri)}\n"
    optional_block += "}\n"

tfvars_content = f"""# ============================================================================
# LLM Backend Onboarding - Generated Terraform Variables
# Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
# Stack:     stacks/llm-backend-onboarding (environment: {env})
# Deploy:    task apply STACK=llm-backend-onboarding ENV={env} VAR_FILE={os.path.relpath(tfvars_file, repo_root)}
# ============================================================================

# Circuit breaker (recommended for production) and Universal LLM API contract
configure_circuit_breaker = true
inference_api_type        = {hcl_str(inference_api_type)}

# ============================================================================
# LLM Backends (backend_mode = "{backend_mode}")
# Each model in supported_models has its own metadata (sku, capacity,
#   modelFormat, modelVersion, retirementDate, apiVersion, timeout,
#   inferenceApiVersion).
# ============================================================================
{backends_block}
# ============================================================================
# Model Aliases (group multiple models under a single client-facing name)
# ============================================================================
{aliases_block}{optional_block}"""

# Write the variables file
utils.print_info(f"Generating Terraform variables file: {tfvars_file}")
utils.write_tfvars(tfvars_file, tfvars_content)

utils.print_ok("Terraform variables file generated successfully!")
print("\n" + "="*60)
print(f"GENERATED {os.path.basename(tfvars_file)} CONTENT:")
print("="*60)
print(tfvars_content)

<a id='6'></a>
### 6️⃣ Deploy LLM Backend Onboarding (Terraform)

Deploy the LLM backends, backend pools, and policy fragments with the `stacks/llm-backend-onboarding` stack. `utils.run_stack` runs `task apply STACK=llm-backend-onboarding ENV=<env> VAR_FILE=<file>` when [Task](https://taskfile.dev) is installed, otherwise the equivalent `terraform -chdir=stacks/llm-backend-onboarding` `init` (with `environments/<env>/backend.hcl`) → `plan` (with `common.tfvars` + the generated file) → `apply`. The stack owns these resources in its own state, so nothing is imported:

In [ ]:
# ============================================================================
# 6️⃣ Deploy the llm-backend-onboarding stack
# ----------------------------------------------------------------------------
# Equivalent to: task apply STACK=llm-backend-onboarding ENV=<env> VAR_FILE=<tfvars_file>
#   terraform -chdir=stacks/llm-backend-onboarding init -reconfigure -backend-config=environments/<env>/backend.hcl
#   terraform -chdir=stacks/llm-backend-onboarding plan -out=tfplan -var-file=environments/<env>/common.tfvars -var-file=<tfvars_file>
#   terraform -chdir=stacks/llm-backend-onboarding apply tfplan
#
# Prerequisites already set by earlier cells: env, repo_root, tfvars_file
# ============================================================================
assert os.path.isfile(tfvars_file), f"tfvars file not found: {tfvars_file}"

result = utils.run_stack("llm-backend-onboarding", env, var_file=tfvars_file, repo_root=repo_root)
if not result.success:
    raise RuntimeError("llm-backend-onboarding deployment failed — see output above.")
utils.print_ok("LLM backend onboarding deployment completed successfully!")

# Stack outputs: terraform -chdir=stacks/llm-backend-onboarding output -json
llm_backend_outputs = utils.stack_outputs("llm-backend-onboarding", env, repo_root=repo_root, refresh=True)
if llm_backend_outputs:
    print("\n" + "=" * 60)
    print("TERRAFORM OUTPUTS:")
    print("=" * 60)
    for _key, _value in llm_backend_outputs.items():
        print(f"  {_key}: {_value}")
else:
    utils.print_warning("Could not retrieve terraform outputs.")

<a id='7'></a>
### 7️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [17]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:45:18.297439 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476
👉🏽 APIM Service Id: /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: 854bc984-355a-4e87-ae7d-1d0eedab998d
👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 2 unique supported models
✅ Updated supported models in APIM policy fragment 'set-backend-pools': ⌚ 14:45:21.282428 
  • gpt-5.4
  • gpt-5.4-mini


In [18]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • content-safety-backend
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
  • foundry-italynorth-0
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
    Models: gpt-5.4, gpt-5.4-mini
  • ms-learn-mcp-backend
    URL: https://learn.microsoft.com/api/mcp

🤖 Total Supported Models: 2
   gpt-5.4, gpt-5.4-mini


<a id='8'></a>
### 8️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [19]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

✅ Policy fragment 'get-available-models' exists! ⌚ 14:46:57.203117 
👉🏽 This fragment returns available model deployments in a format similar to Azure Cognitive Services API.
👉🏽 Description: Returns available model deployments


<a id='test-foundry'></a>
### 🧪 Test GET /deployments (Microsoft Foundry Integration)

Test the `GET /deployments` endpoint which leverages the `get-available-models` policy fragment. This endpoint is used by Microsoft Foundry to discover available model deployments:

In [20]:
# Test GET /deployments endpoint (Microsoft Foundry integration)
# This endpoint uses the get-available-models policy fragment to return available model deployments
# Testing both Universal LLM API and Azure OpenAI API endpoints

def test_get_deployments(base_endpoint, api_name, api_key):
    """Test GET /deployments for a given API endpoint."""
    deployments_url = f"{base_endpoint}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nTesting {api_name} GET /deployments: {deployments_url}")
    
    try:
        response = requests.get(
            deployments_url,
            headers={"api-key": api_key},
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = response.json()
            deployments = data.get("value", [])
            
            utils.print_ok(f"{api_name} GET /deployments returned {len(deployments)} model deployment(s)")
            
            print("\n" + "="*60)
            print(f"AVAILABLE MODEL DEPLOYMENTS ({api_name})")
            print("="*60)
            
            for deployment in deployments:
                print(f"\n📦 Deployment: {deployment.get('name', 'N/A')}")
                print(f"   ID: {deployment.get('id', 'N/A')}")
                print(f"   Type: {deployment.get('type', 'N/A')}")
                
                sku = deployment.get('sku', {})
                if sku:
                    print(f"   SKU: {sku.get('name', 'N/A')} (Capacity: {sku.get('capacity', 'N/A')})")
                
                props = deployment.get('properties', {})
                if props:
                    model = props.get('model', {})
                    if model:
                        print(f"   Model: {model.get('name', 'N/A')} (Format: {model.get('format', 'N/A')}, Version: {model.get('version', 'N/A')})")
                    
                    capabilities = props.get('capabilities', {})
                    if capabilities:
                        caps_list = [k for k, v in capabilities.items() if v == 'true' or v == True]
                        print(f"   Capabilities: {', '.join(caps_list) if caps_list else 'N/A'}")
                    
                    print(f"   Status: {props.get('provisioningState', 'N/A')}")
            
            print("\n" + "="*60)
            utils.print_ok(f"{api_name} GET /deployments test completed successfully!")
            return True
        else:
            utils.print_error(f"{api_name} GET /deployments failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        utils.print_error(f"{api_name} GET /deployments test failed: {str(e)}")
        return False

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

if api_key:
    # Test 1: Universal LLM API - GET /models/deployments
    apimClientTool.discover_api("models")
    azure_endpoint_models = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Universal LLM API Base Endpoint: {azure_endpoint_models}models")
    test_get_deployments(f"{azure_endpoint_models}models/", "Universal LLM API", api_key)
    
    # Test 2: Azure OpenAI API - GET /openai/deployments
    try:
        apimClientTool.discover_api("openai")
        azure_endpoint_openai = str(apimClientTool.azure_endpoint)
        utils.print_info(f"\nAzure OpenAI API Base Endpoint: {azure_endpoint_openai}openai")
        test_get_deployments(f"{azure_endpoint_openai}openai/", "Azure OpenAI API", api_key)
    except Exception as e:
        utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
else:
    utils.print_warning("Cannot test GET /deployments - missing API key")

✅ Using subscription: master ⌚ 14:47:11.506908 
👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Universal LLM API Base Endpoint: https://apim-82da66b0c8.azure-api.net/models
👉🏽 
Testing Universal LLM API GET /deployments: https://apim-82da66b0c8.azure-api.net/models/deployments?api-version=2024-05-01-preview
Response status: 200 - OK
✅ Universal LLM API GET /deployments returned 4 model deployment(s) ⌚ 14:47:15.618109 

AVAILABLE MODEL DEPLOYMENTS (Universal LLM API)

📦 Deployment: gpt-5.4
   ID: aif-citadel-primary
   Type: ai-foundry
   SKU: GlobalStandard (Capacity: 50)
   Model: gpt-5.4 (Format: OpenAI, Version: 2026-03-05)
   Capabilities: chatCompletion
   Status: Succeeded

📦 Deployment: gpt-5.4-mini
   ID: aif-citadel-primary
   Type: ai-foundry
   S

---
## 🧪 Test Deployed Models

The following sections test the deployed models through both the Universal LLM API and Azure OpenAI API endpoints.
---

<a id='test-universal'></a>
### 🧪 Test via Universal LLM API (models/chat/completions)

Test the deployed models using the Universal LLM API which routes based on the `model` field in the request body:

> **Note:** Some models may not support the chat/completions format. Please refer to your LLM backend documentation for supported API formats per model. There is a validation notebook that is designed to test both chat/completions, embeddings, and response formats.

In [21]:
# Discover the Universal LLM API endpoint
apimClientTool.discover_api("models")
azure_endpoint_models = str(apimClientTool.azure_endpoint)
chat_completions_url_models = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Universal LLM API Endpoint: {chat_completions_url_models}")

# Get an API key from subscriptions (it will get the most recently created Access Contract which should be the one we created for testing)
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Universal LLM API Endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
✅ Using subscription: master ⌚ 14:47:35.164087 


In [22]:
# Test each supported model via Universal LLM API
# NOTE: This loop sends a chat/completions request to every model in `updated_supported_models`.
#       Embedding-only models (e.g. text-embedding-*) cannot serve chat/completions and the
#       backend correctly returns HTTP 400 with a body like:
#           "The requested operation is unsupported with the model selected"
#       That outcome proves the request was routed and the backend rejected on its merits, so
#       we treat it as a SUCCESSFUL routing test (not a failure).
def is_unsupported_operation_400(response):
    """Return True for backend 400s that indicate the model doesn't support this API surface
    (typical when chat/completions is sent to an embedding model)."""
    if response.status_code != 400:
        return False
    text = (response.text or "").lower()
    markers = (
        "operation is unsupported",
        "operation_not_supported",
        "operationnotsupported",
        "unsupportedoperation",
        "not supported with the model",
    )
    return any(m in text for m in markers)

if api_key and updated_supported_models:
    utils.print_info(f"\nTesting {len(updated_supported_models)} models via Universal LLM API...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        payload = {
            "model": model_name,
            "messages": test_messages
        }
        
        try:
            utils.print_info(f"Using endpoint: {chat_completions_url_models}")
            
            response = requests.post(
                chat_completions_url_models,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            elif is_unsupported_operation_400(response):
                # Expected for embedding-only models invoked via chat/completions.
                print(f"  ℹ️  Backend returned 400 'operation unsupported' — expected for embedding-only models on chat/completions.")
                utils.print_ok(f"Model '{model_name}' - SUCCESS (routed; chat/completions not supported by this model)\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run tests - missing API key or supported models")


👉🏽 
Testing 2 models via Universal LLM API...

👉🏽 Testing model: gpt-5.4
👉🏽 Using endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
Response status: 200 - OK
  💬 Response: 4
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4' - SUCCESS
 ⌚ 14:47:52.163684 
👉🏽 Testing model: gpt-5.4-mini
👉🏽 Using endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
Response status: 200 - OK
  💬 Response: Four
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4-mini' - SUCCESS
 ⌚ 14:47:54.306366 


<a id='test-openai'></a>
### 🧪 Test via Azure OpenAI API (openai/deployments/{model}/chat/completions)

Test the deployed models using the Azure OpenAI compatible API which uses the deployment name in the URL path:

In [23]:
# Discover the Azure OpenAI API endpoint
try:
    apimClientTool.discover_api("openai")
    azure_endpoint_openai = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Azure OpenAI API Base Endpoint: {azure_endpoint_openai}")
except Exception as e:
    utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
    azure_endpoint_openai = None

👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/azure-openai-api and path openai
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Azure OpenAI API Base Endpoint: https://apim-82da66b0c8.azure-api.net/


In [24]:
# Test models via Azure OpenAI API format
# NOTE: Embedding-only models will return HTTP 400 "operation is unsupported" when invoked
#       through chat/completions. We treat that as a SUCCESSFUL routing test — the helper
#       `is_unsupported_operation_400` (defined in the Universal LLM API test cell above) is
#       reused here.
if api_key and azure_endpoint_openai and updated_supported_models:
    utils.print_info(f"\nTesting models via Azure OpenAI API format...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        # Azure OpenAI format uses deployment name in URL path
        chat_completions_url_openai = f"{azure_endpoint_openai}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
        
        payload = {
            "messages": test_messages  # No model field needed - it's in the URL
        }
        
        try:
            response = requests.post(
                chat_completions_url_openai,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            elif is_unsupported_operation_400(response):
                # Expected for embedding-only models invoked via chat/completions.
                print(f"  ℹ️  Backend returned 400 'operation unsupported' — expected for embedding-only models on chat/completions.")
                utils.print_ok(f"Model '{model_name}' - SUCCESS (routed; chat/completions not supported by this model)\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run Azure OpenAI API tests - missing API key, endpoint, or supported models")


👉🏽 
Testing models via Azure OpenAI API format...

👉🏽 Testing model: gpt-5.4
Response status: 200 - OK
  💬 Response: Paris
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4' - SUCCESS
 ⌚ 14:48:58.068028 
👉🏽 Testing model: gpt-5.4-mini
Response status: 200 - OK
  💬 Response: Paris
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4-mini' - SUCCESS
 ⌚ 14:49:00.088566 


<a id='test-sdk'></a>
### 🧪 Test using Azure OpenAI Python SDK

Test using the official Azure OpenAI Python SDK:

In [25]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict.
    The AzureOpenAI SDK sends both api-key and Authorization: Bearer headers,
    but APIM tries to validate the Authorization header as a JWT token."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing with Azure OpenAI SDK using model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from Azure OpenAI SDK!'"}
            ]
        )
        
        utils.print_ok("SDK Test Successful!")
        print(f"💬 Response: {response.choices[0].message.content}")
        print(f"📊 Usage: {response.usage.total_tokens} tokens")
        
    except Exception as e:
        utils.print_error(f"SDK Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run SDK test - missing prerequisites")

👉🏽 Testing with Azure OpenAI SDK using model: gpt-5.4
✅ SDK Test Successful! ⌚ 14:49:15.178711 
💬 Response: Hello from Azure OpenAI SDK!
📊 Usage: 35 tokens


<a id='test-streaming'></a>
### 🧪 Test Streaming Response

Test streaming responses using the Azure OpenAI SDK:

In [26]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing streaming with model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        start_time = time.time()
        
        response = client.chat.completions.with_raw_response.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Count from 1 to 10 with commas between each number."}
            ],
            stream=True
        )
        
        print(f"📡 x-ms-region: {response.headers.get('x-ms-region', 'unknown')}")
        print(f"📡 x-ms-stream: {response.headers.get('x-ms-stream', 'N/A')}")
        print("\n💬 Streaming response:")
        
        completion = response.parse()
        collected_content = []
        total_chunks = 0
        content_chunks = 0
        first_chunk_time = None
        
        for chunk in completion:
            total_chunks += 1
            if first_chunk_time is None:
                first_chunk_time = time.time()
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                collected_content.append(content)
                content_chunks += 1
                print(content, end='', flush=True)
        
        elapsed = time.time() - start_time
        ttfb = (first_chunk_time - start_time) if first_chunk_time else None
        print(f"\n\n✅ Stream completed in {elapsed:.2f} seconds")
        print(f"📦 Chunks received: {total_chunks} (with content: {content_chunks})")
        if ttfb is not None:
            print(f"⚡ Time to first chunk: {ttfb:.2f} seconds")
        print(f"📝 Full response: {''.join(collected_content)}")
        
    except Exception as e:
        utils.print_error(f"Streaming Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run streaming test - missing prerequisites")


👉🏽 Testing streaming with model: gpt-5.4
📡 x-ms-region: Italy North
📡 x-ms-stream: N/A

💬 Streaming response:
1, 2, 3, 4, 5, 6, 7, 8, 9, 10

✅ Stream completed in 2.68 seconds
📦 Chunks received: 31 (with content: 28)
⚡ Time to first chunk: 2.41 seconds
📝 Full response: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10


<a id='test-aliases'></a>
### 🧪 Test Model Aliases (consistent across Universal LLM, Azure OpenAI, Unified AI)

If you configured `model_aliases` in the initialization cell, this section sends a request using each **alias name** (instead of a real model) against:

- **Universal LLM API** — `model` field in the request body
- **Azure OpenAI API** — alias in the URL path (`/deployments/{alias}/chat/completions`)
- **Unified AI API** (if deployed) — wildcard route, `model` in body

The gateway's shared `resolve-model-alias` policy fragment translates each alias into a real underlying model based on the configured `priority` or `weighted` strategy. The cell below validates the resolution by inspecting the response payload and (when available) the `UAIG-Model-Id` debug header on Unified AI.

> Direct model selection still works unchanged — the alias resolver is a no-op when the requested model is not an alias name.


In [27]:
# Test that each configured model alias resolves correctly across both APIs.
# - Universal LLM API: alias is sent in the request body's `model` field.
# - Azure OpenAI API: alias is in the URL path (/deployments/{alias}/chat/completions).
# Both should return a successful chat completion. The response model name reflects
# the resolved underlying model (the gateway rewrites `model` in the body before forwarding).
#
# An alias that resolves to an embedding-only model on chat/completions will get HTTP 400
# "operation is unsupported" from the backend — that still proves the alias was resolved
# and routed correctly, so we treat it as a SUCCESSFUL routing test using
# `is_unsupported_operation_400` (defined in the per-model Universal LLM API test cell above).

if not model_aliases:
    utils.print_warning("No model aliases configured — skipping alias resolution tests.")
elif not api_key:
    utils.print_warning("No API key available — skipping alias resolution tests.")
else:
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "Reply with the single word OK."}
    ]

    utils.print_info(f"\nTesting {len(model_aliases)} model aliases via the Universal LLM API and Azure OpenAI API...\n")

    for alias in model_aliases:
        alias_name = alias["name"]
        strategy = alias.get("strategy", "priority")
        underlying = ", ".join(alias.get("models", []))
        weights = alias.get("weights")
        weights_str = f" weights={weights}" if weights else ""
        print(f"\n🔁 Alias: {alias_name}  (strategy: {strategy}{weights_str})")
        print(f"   → underlying models: [{underlying}]")

        # ---- Universal LLM API ----
        try:
            url_universal = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"
            r = requests.post(
                url_universal,
                headers={"api-key": api_key},
                json={"model": alias_name, "messages": test_messages},
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                resolved_model = data.get("model", "?")
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                utils.print_ok(f"   Universal LLM API → resolved model: '{resolved_model}'  reply: '{answer.strip()}'")
            elif is_unsupported_operation_400(r):
                utils.print_ok(f"   Universal LLM API → routed (alias resolved); backend returned 400 'operation unsupported' — expected for embedding-only target")
            else:
                utils.print_error(f"   Universal LLM API → {r.status_code}: {r.text[:200]}")
        except Exception as e:
            utils.print_error(f"   Universal LLM API → ERROR: {e}")

        # ---- Azure OpenAI API ----
        if azure_endpoint_openai:
            try:
                url_openai = f"{azure_endpoint_openai}openai/deployments/{alias_name}/chat/completions?api-version={inference_api_version}"
                r = requests.post(
                    url_openai,
                    headers={"api-key": api_key},
                    json={"messages": test_messages},
                    timeout=60,
                )
                if r.status_code == 200:
                    data = r.json()
                    resolved_model = data.get("model", "?")
                    answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                    utils.print_ok(f"   Azure OpenAI API   → resolved model: '{resolved_model}'  reply: '{answer.strip()}'")
                elif is_unsupported_operation_400(r):
                    utils.print_ok(f"   Azure OpenAI API   → routed (alias resolved); backend returned 400 'operation unsupported' — expected for embedding-only target")
                else:
                    utils.print_error(f"   Azure OpenAI API   → {r.status_code}: {r.text[:200]}")
            except Exception as e:
                utils.print_error(f"   Azure OpenAI API   → ERROR: {e}")

        # ---- Unified AI API (optional; only if the API exists) ----
        try:
            apimClientTool.discover_api("unified-ai")
            azure_endpoint_unified = str(apimClientTool.azure_endpoint)
            url_unified = f"{azure_endpoint_unified}models/chat/completions?api-version={inference_api_version}"
            r = requests.post(
                url_unified,
                headers={"api-key": api_key, "UAIG-Config-Cache-Bypass": "true"},
                json={"model": alias_name, "messages": test_messages},
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                resolved_model = data.get("model", "?")
                uaig_model = r.headers.get("UAIG-Model-Id", "(header not enabled)")
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                utils.print_ok(f"   Unified AI API     → resolved model: '{resolved_model}' (UAIG-Model-Id={uaig_model})  reply: '{answer.strip()}'")
            elif is_unsupported_operation_400(r):
                uaig_model = r.headers.get("UAIG-Model-Id", "(header not enabled)")
                utils.print_ok(f"   Unified AI API     → routed (UAIG-Model-Id={uaig_model}); backend returned 400 'operation unsupported' — expected for embedding-only target")
            else:
                utils.print_error(f"   Unified AI API     → {r.status_code}: {r.text[:200]}")
        except Exception as e:
            # API not deployed or other error — non-fatal for this validation
            utils.print_warning(f"   Unified AI API     → skipped: {str(e)[:120]}")


👉🏽 
Testing 2 model aliases via the Universal LLM API and Azure OpenAI API...


🔁 Alias: adv-gpt  (strategy: priority)
   → underlying models: [gpt-5.4]
✅    Universal LLM API → resolved model: 'gpt-5.4-2026-03-05'  reply: 'OK' ⌚ 14:49:32.982322 
✅    Azure OpenAI API   → resolved model: 'gpt-5.4-2026-03-05'  reply: 'OK' ⌚ 14:49:35.938887 
👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/unified-ai-api and path unified-ai
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
✅    Unified AI API     → resolved model: 'gpt-5.4-2026-03-05' (UAIG-Model-Id=(header not enabled))  reply: 'OK' ⌚ 14:49:39.479403 

🔁 Alias: mini-gpt  (strategy: priority)
   → underlying models: [gpt-5.4-mini]
✅    Universal LLM API → resolved model: 'gpt-5.4-mini-2026-03-17'  reply: 'OK' ⌚ 14:49:41.592556 
✅    Azure OpenAI API   → resolved model: 'gpt-5.4-mini-2026-03-17'  

---
## 📊 Results Summary
---

This notebook completed the following tasks:

1. **Extracted** current APIM backend-pools configurations
2. **Generated** the LLM backend stack variables file (`environments/<env>/llm-backend-onboarding.tfvars`) supporting:
   - Foundry-derived backends (`foundry_backends`), a full override (`llm_backend_config`) or extras (`extra_llm_backends`)
   - Backend types: `ai-foundry`, `azure-openai`, `aws-bedrock`, `aws-bedrock-mantle`, `gemini-openai`, `external`
   - Auth: legacy `auth_scheme` (`managedIdentity` | `apiKey` | `token`) and/or `auth_type` (`managed-identity`, `aws-sigv4`, `api-key-bearer`, `api-key-header`, `none`)
   - `auth_config` with optional **Key Vault references** (`key_vault_secret_uri`) for API-key backends
   - Per-model metadata (`sku`, `capacity`, `modelFormat`, `modelVersion`, `retirementDate`, `apiVersion`, `timeout`, `inferenceApiVersion`)
   - **Model aliases** (`model_aliases`) with `priority` or `weighted` strategies — same alias map honored consistently across Azure OpenAI, Universal LLM, and Unified AI APIs
   - **AWS credentials** (`aws = { region, access_key_secret_uri, secret_key_secret_uri }`, Key Vault references) for Bedrock
3. **Deployed** the [`../stacks/llm-backend-onboarding`](../stacks/llm-backend-onboarding) stack (`task apply STACK=llm-backend-onboarding ENV=<env>` / `terraform -chdir=stacks/llm-backend-onboarding` init → plan → apply), which creates the shared `resolve-model-alias` policy fragment alongside backends, pools, and other fragments
4. **Verified** deployment including the `get-available-models` policy fragment
5. **Tested** the deployed models through:
   - **GET /deployments** (Microsoft Foundry integration — uses `get-available-models` policy fragment)
   - Universal LLM API (`/models/chat/completions`)
   - Azure OpenAI API (`/openai/deployments/{model}/chat/completions`)
   - Azure OpenAI Python SDK
   - Streaming responses
   - **Model alias resolution** across Universal LLM API, Azure OpenAI API, and Unified AI API (when configured)

### Next Steps

- Modify the `llm_backends_config` in the initialization cell to add more backends (Azure OpenAI, Bedrock, Bedrock Mantle, Gemini, external)
- For API-key backends, set `authConfig.keyVaultSecretUri` (versionless secret URI) to source secrets from Azure Key Vault (recommended over inline `secretValue`)
- Tune `model_aliases` to expose simplified client-facing names (e.g. `adv-gpt`, `fast-gpt`, `reasoning`) that route to one or more underlying models — `priority` strategy gives automatic cross-model fallback (Unified AI), `weighted` strategy enables A/B testing of model swaps
- Store the AWS keys in Key Vault and provide `aws_access_key_secret_uri` / `aws_secret_key_secret_uri` / `aws_region` (written to the `aws` object) when onboarding `aws-bedrock` (SigV4) backends
- Re-run the generate + deploy cells to update the APIM configuration, or run `task apply STACK=llm-backend-onboarding ENV=<env>` from the repository root
- Commit `environments/<env>/llm-backend-onboarding.tfvars` so CI/CD (`task plan-all` / `task up`) deploys the same configuration
- Use the `get-available-models` policy fragment to expose available models to Microsoft Foundry and other clients


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

The backends and policy fragments deployed by this notebook are intended to persist as part of your Governance Hub configuration. To remove them, delete the corresponding backend resources and policy fragments from your APIM instance via the Azure portal or CLI.

> **Note:** Removing backends will affect any access contracts that reference those backend pools. Ensure no active contracts depend on them before cleanup.